# Modèle v3 — clustering territorial et lissage bayésien

Le diagnostic du v1 était sans ambiguïté : **54,6 % de l'importance partait de
l'historique de la commune**. Le modèle disait surtout « ce qui a brûlé
rebrûlera ».

Ce n'est pas une fuite — le jour J on connaît réellement le passé — mais ça
laissait un trou : **une commune qui n'a jamais brûlé gardait un score bas,
même entourée de communes qui brûlent chaque été.** C'est le problème classique
de *small area estimation* : trop peu d'événements pour estimer un taux commune
par commune.

Ce notebook mesure la parade.

---

### Le principe

Regrouper les communes qui se ressemblent **physiquement**, estimer le taux sur
le groupe, et faire retomber chaque commune vers le taux de son groupe à
proportion de ce qu'on sait d'elle.

    p_cluster = (feux_c + k0 · p_national) / (jours_c + k0)
    p_commune = (feux_i + k1 · p_cluster)  / (jours_i + k1)

Une commune avec beaucoup de feux garde son taux propre. Une commune sans
historique hérite du risque de ses semblables, au lieu d'hériter de zéro.

### Les quatre colonnes ajoutées

| Colonne | Sens |
|---|---|
| `cluster_id` | le groupe territorial de la commune |
| `taux_cluster_lisse` | le risque de fond de ce type de territoire |
| `taux_commune_lisse` | le risque de la commune, rappelé vers son cluster |
| `ratio_commune_cluster` | la commune brûle-t-elle plus ou moins que ses pairs |

Tout le reste est **identique au v2** : mêmes features météo, végétation et
calendrier, mêmes hyperparamètres Optuna. L'expérience est donc propre.

---

### ⚠️ Les quatre garde-fous anti-fuite

Le lissage agrège `y`. C'est la feature la plus exposée du projet : une erreur
ici ne lève rien, elle produit un excellent score de train et un modèle qui
s'effondre en production.

1. **Le profil ne lit ni la grille ni les feux.** Un clustering construit sur
   la sinistralité serait circulaire : on prédirait le feu avec des groupes
   définis par le feu. Il n'utilise que CORINE **millésime 2006** (≤ toutes les
   dates du jeu) et la climatologie FWI bornée à **2006-2019**.

2. **Les taux sont agrégés sur le train COMPLET** — 177,6 M lignes — jamais sur
   le train échantillonné, où le taux vaut 9,1 % contre 0,0189 % en réalité.
   Un facteur **×487** sur le prior, que rien dans les métriques ne signalerait.

3. **Pour une ligne de train de l'année Y, les taux excluent l'année Y.** Sans
   ça, une ligne de 2012 contribuerait à sa propre feature — la fuite classique
   du target encoding.

4. **L'exclusion se fait à exposition constante.** Retirer une année fait
   tomber le dénominateur de 5 113 à 4 748 jours ; sans correction la même
   commune recevait un taux 4,3 % plus élevé en train qu'en validation, et
   `ratio_commune_cluster` prenait deux plages *disjointes*. Pas une fuite,
   mais un décalage train/service invisible dans les métriques d'entraînement.
   Les comptages sont donc extrapolés à l'exposition complète.

Quatorze tests dans `tests/test_clustering.py` verrouillent ces règles.

## 1. Ce que le clustering a produit

In [ ]:
# ── enregistrement automatique des figures ──
# chaque plt.show() écrit aussi un PNG dans figures/modele-v3/
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "src" / "tvfed").is_dir():
        sys.path.insert(0, str(_p / "src"))
        break

from tvfed.figures import activer

activer("modele-v3")

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for p in (Path.cwd(), *Path.cwd().parents):
    if (p / "src" / "tvfed").is_dir():
        sys.path.insert(0, str(p / "src"))
        RACINE = p
        break

from tvfed import clustering as CLU

# charte identique aux notebooks d'audit
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
BLEU, ORANGE, ROUGE, VERT, VIOLET = "#2a78d6", "#eb6834", "#e34948", "#1baf7a", "#4a3aa7"
GRIS = "#c3c2b7"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
                     "font.size": 9, "axes.edgecolor": "#c3c2b7", "text.color": INK})

PROC = RACINE / "data" / "processed"
BASE = pd.read_csv(PROC / "baselines.csv")

PROFIL = pd.read_parquet(PROC / "profil_communes.parquet")
TAUX = pd.read_parquet(PROC / "taux_lisses.parquet")
CFG = json.loads((PROC / "clustering.json").read_text(encoding="utf-8"))
SIN = CLU.sinistralite()          # feux et jours par commune × année, train seul

print(f"{len(PROFIL):,} communes profilées sur {PROFIL.shape[1] - 2} variables")
print(f"{PROFIL.cluster_id.nunique()} clusters ({CFG['methode']}), "
      f"k0={CFG['k0_cluster']:,} k1={CFG['k1_commune']:,}, "
      f"poids position {CFG['poids_position']}")
print(f"{SIN.jours.sum():,} jours-commune de train, {SIN.feux.sum():,} feux "
      f"→ taux national {SIN.feux.sum() / SIN.jours.sum():.6%}")


## 2. La typologie territoriale

Trente groupes, formés sur la végétation, le relief, la densité humaine et la
climatologie du FWI. **Jamais sur le feu.**

`lat` et `lon` entrent dans le calcul mais à **25 % du poids** des autres
variables. Sans eux, les groupes seraient éclatés d'un bout à l'autre du pays ;
à poids plein, ils dégénéreraient en pavés géographiques et le clustering ne
serait qu'un découpage administratif déguisé.

La sinistralité n'intervient qu'après coup, pour donner à chaque groupe son
taux. C'est donc autant une vérification qu'une illustration : si des groupes
formés sans regarder le feu se retrouvent à avoir des risques très différents,
c'est que la composition du territoire porte bien du signal.

In [ ]:
"""FIG 1 — Les 30 types de territoire, et ce qui les distingue.

Le clustering regroupe les communes sur leurs caractéristiques PHYSIQUES —
végétation, relief, densité humaine, climatologie du FWI. Jamais sur le feu.
La sinistralité n'entre qu'après, pour donner à chaque groupe son taux.

C'est donc une vérification autant qu'une illustration : si les groupes formés
sans regarder le feu se retrouvent à avoir des risques très différents, c'est
que la composition du territoire porte bien du signal.
"""
REF = TAUX[TAUX.an_exclue == 0].set_index("code_insee")
P = PROFIL.join(REF[["taux_cluster_lisse"]])

# risque de chaque cluster, en % de chances de feu un jour donné
risque = (P.groupby("cluster_id").taux_cluster_lisse.first() * 100
          ).sort_values(ascending=False)
ordre = risque.index.to_list()

fig = plt.figure(figsize=(15.5, 6.6))
gs = fig.add_gridspec(1, 3, width_ratios=[.92, 1.62, .70], wspace=.42)

# ── (a) la carte ─────────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0])
sc = ax0.scatter(P.lon, P.lat, c=P.taux_cluster_lisse * 100, s=1.1,
                 cmap="YlOrRd", vmin=0, vmax=risque.max(), edgecolors="none")
ax0.set_aspect(1 / np.cos(np.radians(46.5)))
ax0.set_xticks([]); ax0.set_yticks([])
ax0.spines[:].set_visible(False)
ax0.set_title("Le risque de fond", fontsize=11.5, weight="bold",
              loc="left", y=1.0)
cb = fig.colorbar(sc, ax=ax0, fraction=.040, pad=.02,
                  orientation="horizontal", location="bottom")
cb.set_label("risque quotidien du cluster (%)", fontsize=8.5)
cb.ax.xaxis.set_label_position("bottom")
cb.ax.tick_params(labelsize=8, colors=MUTED)
cb.outline.set_visible(False)

# ── (b) ce qui caractérise chaque cluster ────────────────────────────────
ax1 = fig.add_subplot(gs[1])
VARS = {
    "part_maquis": "maquis",
    "part_coniferes": "conifères",
    "part_feuillus": "feuillus",
    "part_landes": "landes",
    "part_agricole": "agricole",
    "part_artificialise": "urbanisé",
    "altitude_moy": "altitude",
    "amplitude_altitude": "relief",
    "log_densite": "densité hab.",
    "distance_cote_km": "dist. côte",
    "fwi_moyen": "FWI moyen",
    "jours_fwi_sup_21": "j. FWI > 21",
}
M = P.groupby("cluster_id")[list(VARS)].mean().loc[ordre]
# centré-réduit par variable : on lit un écart au territoire français moyen,
# pas une valeur absolue (comparer un % de maquis à une altitude n'a pas de sens)
Z = (M - M.mean()) / M.std()

im = ax1.imshow(Z.to_numpy(), cmap="RdBu_r", vmin=-2.2, vmax=2.2, aspect="auto")
ax1.set_xticks(range(len(VARS)))
ax1.set_xticklabels(VARS.values(), rotation=45, ha="right", fontsize=8.5)
ax1.set_yticks(range(len(ordre)))
ax1.set_yticklabels([f"c{c}" for c in ordre], fontsize=7)
ax1.set_title("Ce qui caractérise chaque groupe (écart à la moyenne)",
              fontsize=11.5, weight="bold", loc="left")
ax1.tick_params(colors=MUTED, length=0)
ax1.spines[:].set_visible(False)
cb1 = fig.colorbar(im, ax=ax1, fraction=.024, pad=.015)
cb1.set_label("écarts-types", fontsize=8.5)
cb1.ax.tick_params(labelsize=8, colors=MUTED)
cb1.outline.set_visible(False)

# ── (c) le risque, dans le même ordre ────────────────────────────────────
ax2 = fig.add_subplot(gs[2])
n_com = P.cluster_id.value_counts().reindex(ordre)
b = ax2.barh(range(len(ordre)), risque.to_numpy(), color=ROUGE,
             edgecolor="#fcfcfb", linewidth=.6, height=.78)
for i, (r, n) in enumerate(zip(risque, n_com)):
    ax2.text(r + risque.max() * .03, i, f"{n:,}", va="center",
             fontsize=6.5, color=MUTED)
ax2.set_yticks([]); ax2.invert_yaxis()
ax2.set_xlim(0, risque.max() * 1.34)
ax2.set_xlabel("risque quotidien (%)", fontsize=8.5)
ax2.set_title("Risque\n(et nb de communes)", fontsize=11.5, weight="bold", loc="left")
ax2.grid(axis="x", color=GRID, lw=.7); ax2.set_axisbelow(True)
ax2.spines[["top", "right", "left"]].set_visible(False)
ax2.tick_params(colors=MUTED)

fig.suptitle(f"Typologie territoriale — {len(ordre)} groupes formés SANS regarder le feu, "
             f"{len(P):,} communes",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.0)
plt.show()

haut, bas = ordre[0], ordre[-1]
print(f"Cluster le plus exposé  : c{haut}  {risque.iloc[0]:.4f} % "
      f"({n_com.iloc[0]:,} communes)")
print(f"Cluster le moins exposé : c{bas}  {risque.iloc[-1]:.4f} % "
      f"({n_com.iloc[-1]:,} communes)")
print(f"Écart : ×{risque.iloc[0] / risque.iloc[-1]:.0f}\n")
print(f"Ce qui distingue c{haut} (en écarts-types) :")
for v, z in Z.loc[haut].sort_values(key=abs, ascending=False).head(5).items():
    print(f"   {VARS[v]:>14s}  {z:+.2f}")
print(f"\nCe qui distingue c{bas} :")
for v, z in Z.loc[bas].sort_values(key=abs, ascending=False).head(5).items():
    print(f"   {VARS[v]:>14s}  {z:+.2f}")
print()
print("→ Les groupes ont été formés sans jamais regarder où le feu tombe.")
print(f"  Qu'ils se retrouvent à ×{risque.iloc[0] / risque.iloc[-1]:.0f} d'écart de risque n'est donc pas")
print("  une tautologie : c'est la mesure de ce que la COMPOSITION du")
print("  territoire explique, à elle seule, avant toute météo.")


## 3. Le lissage, commune par commune

Le lissage fait **deux choses en même temps**, et c'est ce qui se voit sur les
deux panneaux ci-dessous :

- il **relève** les communes muettes, qui valaient 0 faute de données ;
- il **rabat** les communes à l'historique chargé, dont le taux brut repose sur
  une poignée d'événements en quatorze ans.

Dans les deux cas, il remplace un **comptage** par une **estimation**.

In [ ]:
"""FIG 2 — Le trou que le clustering vient boucher.

Sur 2006-2019, **80,4 % des communes n'ont jamais brûlé**. Pour le modèle v1
elles étaient rigoureusement identiques : zéro feu sur 7 jours, zéro sur 30,
zéro sur 90, zéro sur 365. Quatre features, quatre zéros, pour 27 938 communes
qui vont pourtant de la garrigue varoise au bocage normand.

Le lissage bayésien leur rend une estimation : celle des communes qui leur
ressemblent physiquement.
"""
T = pd.read_parquet(PROC / "taux_lisses.parquet")
S = SIN.groupby("code_insee").feux.sum()
JOURS = SIN.groupby("code_insee").jours.sum()
REF = T[T.an_exclue == 0].set_index("code_insee")
MUETTES = S[S == 0].index
PARLANTES = S[S > 0].index

fig, ax = plt.subplots(1, 2, figsize=(14, 5.1))

# ── (a) les muettes, classées par risque ────────────────────────────────
# ⚠️ Ces communes ont TOUTES le même nombre de jours de train (la grille est
# dense et rectangulaire) et toutes zéro feu. Leur taux lissé se réduit donc
# exactement au prior de leur cluster : il n'existe que 30 valeurs distinctes
# pour 27 938 communes. Ni histogramme ni déciles n'ont de sens ici — la
# courbe cumulée montre la vraie structure, en escalier.
tm = REF.loc[MUETTES, "taux_commune_lisse"].sort_values() * 100
part = np.arange(1, len(tm) + 1) / len(tm) * 100

ax[0].step(part, tm.to_numpy(), where="post", lw=2.6, color=VIOLET)
ax[0].fill_between(part, 0, tm.to_numpy(), step="post", color=VIOLET, alpha=.14)
ax[0].axhline(0, color=ROUGE, lw=2.4)
ax[0].text(2, tm.max() * .055, "← le v1 les voyait toutes ici, à zéro",
           fontsize=9.5, color=ROUGE, weight="bold")
ax[0].set_xlabel("part des communes muettes, classées par risque croissant (%)")
ax[0].set_ylabel("risque quotidien estimé (%)")
ax[0].set_xlim(0, 100)
ax[0].set_ylim(-tm.max() * .03, tm.max() * 1.12)
ax[0].set_title(f"Les {len(MUETTES):,} communes qui n'ont jamais brûlé",
                fontsize=11.5, weight="bold", loc="left")
ax[0].text(4, tm.max() * .74,
           f"{len(MUETTES):,} communes, {tm.nunique()} valeurs distinctes :\n"
           f"n'ayant aucun feu et toutes le même nombre\n"
           f"de jours, chacune hérite exactement du taux\n"
           f"de son cluster — d'où l'escalier\n\n"
           f"la plus exposée vaut ×{tm.max() / tm.min():.0f} la moins exposée",
           fontsize=9.5, color=INK, va="top",
           bbox=dict(boxstyle="round,pad=.6", fc="#fcfcfb", ec=GRID))

# ── (b) le lissage tire dans les deux sens ──────────────────────────────
brut = (S / JOURS).loc[PARLANTES] * 100
lisse = REF.loc[PARLANTES, "taux_commune_lisse"] * 100
ax[1].scatter(brut, lisse, s=9, alpha=.30, color=BLEU, edgecolors="none",
              label=f"{len(PARLANTES):,} communes ayant brûlé")
ax[1].scatter(np.zeros(len(MUETTES)), REF.loc[MUETTES, "taux_commune_lisse"] * 100,
              s=9, alpha=.22, color=VIOLET, edgecolors="none",
              label=f"{len(MUETTES):,} communes muettes (brut = 0)")

lim = brut.max() * 1.04
ax[1].plot([0, lim], [0, lim], color=INK, ls="--", lw=1.4)
ax[1].text(lim * .78, lim * .82, "sans lissage (y = x)", fontsize=9,
           color=MUTED, rotation=45, rotation_mode="anchor")
ax[1].set_xlim(-lim * .015, lim)
ax[1].set_ylim(-lim * .015, lim)
ax[1].set_xlabel("risque BRUT — feux observés / jours observés (%)")
ax[1].set_ylabel("risque LISSÉ (%)")
ax[1].set_title("Ce que le lissage change, commune par commune",
                fontsize=11.5, weight="bold", loc="left")
ax[1].legend(frameon=False, fontsize=9, loc="upper left", markerscale=2.5)
ax[1].text(lim * .46, lim * .30,
           "tous les points sont SOUS la diagonale :\n"
           "le lissage rabat les communes à l'historique\n"
           "chargé — quelques feux en quatorze ans ne\n"
           "font pas une probabilité fiable",
           fontsize=9, color=INK, va="top",
           bbox=dict(boxstyle="round,pad=.6", fc="#fcfcfb", ec=GRID))

for a in ax:
    a.grid(color=GRID, lw=.7); a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False); a.tick_params(colors=MUTED)
fig.suptitle("Le lissage bayésien — rendre une estimation aux communes sans historique",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.0)
plt.tight_layout(); plt.show()

print(f"{len(MUETTES):,} communes sans aucun feu sur 2006-2019 "
      f"({len(MUETTES) / len(S):.1%})")
print(f"  {tm.nunique()} valeurs distinctes seulement — une par cluster")
print(f"  la plus faible : {tm.min():.5f} %")
print(f"  la plus forte  : {tm.max():.5f} %")
print(f"  écart          : ×{tm.max() / tm.min():.0f}\n")
print(f"{len(PARLANTES):,} communes ont brûlé au moins une fois")
print(f"  risque brut médian  : {brut.median():.4f} %")
print(f"  risque lissé médian : {lisse.median():.4f} %")
print(f"  la plus chargée passe de {brut.max():.3f} % à "
      f"{lisse[brut.idxmax()]:.3f} % "
      f"(−{100 * (1 - lisse[brut.idxmax()] / brut.max()):.0f} %)\n")
print("→ Le lissage fait DEUX choses en même temps :")
print("  · il RELÈVE les communes muettes, qui valaient 0 faute de données")
print("  · il RABAT les communes à l'historique chargé, dont le taux brut")
print("    repose sur une poignée d'événements en quatorze ans")
print("  Dans les deux cas il remplace un comptage par une ESTIMATION —")
print("  c'est exactement ce que le v1 ne savait pas faire.")


## 4. Ce que ça rapporte

Mesure sur la validation intégrale 2020-2022 — 38 068 464 lignes, 9 176 feux.
Cette partition n'a servi ni à choisir les hyperparamètres (Optuna a travaillé
sur un découpage interne au train) ni à choisir le nombre de clusters.

⚠️ **Sur le choix de k.** Six configurations ont été comparées sur le découpage
interne : toutes tiennent dans 0,0016 de PR-AUC, c'est-à-dire du bruit. Ce
découpage évalue sur du train échantillonné à ~8 % de positifs, alors que le
clustering agit sur l'extrême queue du classement — il est structurellement
aveugle à ce qu'on cherche à mesurer. Le choix de **k = 30** est donc un
arbitrage de fond (80 % des communes n'ont jamais brûlé, et pour elles la
résolution du cluster fait toute la différence), pas une conclusion tirée des
chiffres.

In [ ]:
"""FIG 3 — Ce que le clustering a rapporté, et d'où vient le gain.

Mesure sur la validation intégrale 2020-2022 : 38 M lignes, 9 176 feux.
Cette partition n'a servi ni à choisir les hyperparamètres (Optuna a travaillé
sur un découpage interne au train) ni à choisir le nombre de clusters.
"""
V1 = pd.read_csv(PROC / "modeles_v1.csv").set_index("modele")
V2 = pd.read_csv(PROC / "modeles_v2.csv")
V3 = pd.read_csv(PROC / "modeles_v3.csv")
IMP3 = pd.read_csv(PROC / "importances_v3.csv", index_col=0).squeeze("columns")
CMP = pd.read_csv(PROC / "comparaison_clusters.csv")

BASE_MAX = BASE.pr_auc.max()
NEUVES = ["taux_commune_lisse", "taux_cluster_lisse",
          "ratio_commune_cluster", "cluster_id"]
JOLI = {"taux_commune_lisse": "risque lissé\nde la commune",
        "taux_cluster_lisse": "risque du\ncluster",
        "ratio_commune_cluster": "écart commune\n/ cluster",
        "cluster_id": "identifiant\ndu cluster"}

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))

# ── (a) la progression complète ──────────────────────────────────────────
etapes = pd.DataFrame({
    "nom": ["Meilleure\nbaseline", "XGBoost v1\n(à la main)",
            "XGBoost v2\n(Optuna)", "XGBoost v3\n(+ clustering)"],
    "v": [BASE_MAX, V1.pr_auc["XGBoost"], V2.pr_auc[0], V3.pr_auc[0]],
})
coul = ["#86b6ef", BLEU, VIOLET, VERT]
b = ax[0].bar(etapes.nom, etapes.v, color=coul, edgecolor="#fcfcfb",
              linewidth=1.2, width=.62)
for r, v in zip(b, etapes.v):
    ax[0].text(r.get_x() + r.get_width() / 2, v + etapes.v.max() * .025,
               f"{v:.4f}", ha="center", fontsize=10.5, weight="bold")
    ax[0].text(r.get_x() + r.get_width() / 2, v * .5, f"×{v / BASE_MAX:.2f}",
               ha="center", fontsize=10, color="#fcfcfb", weight="bold")
ax[0].axhline(BASE_MAX, color=INK, ls="--", lw=1.2)
ax[0].set_ylabel("PR-AUC sur la validation intégrale")
ax[0].set_ylim(0, etapes.v.max() * 1.20)
ax[0].set_title("La progression, de bout en bout", fontsize=11.5,
                weight="bold", loc="left")
ax[0].tick_params(axis="x", labelsize=8.5)

# ── (b) le clustering a-t-il AJOUTÉ, ou seulement REMPLACÉ ? ─────────────
# La question décisive. `taux_commune_lisse` arrive n°1 sur 52 features, et
# pourtant la PR-AUC ne bouge que de +1,3 %. La réponse est dans le transfert
# d'importance : il se sert d'abord sur `feux_commune_365j`.
IMP1 = pd.read_csv(PROC / "importances_v1.csv").set_index("feature")["xgb"]
HIST = ["feux_commune_365j", "jours_depuis_dernier_feu", "feux_commune_90j",
        "feux_commune_30j", "feux_commune_7j"]
ETIQ = {"feux_commune_365j": "feux 365 j", "jours_depuis_dernier_feu": "j. depuis dernier",
        "feux_commune_90j": "feux 90 j", "feux_commune_30j": "feux 30 j",
        "feux_commune_7j": "feux 7 j",
        "taux_commune_lisse": "risque lissé commune",
        "ratio_commune_cluster": "écart commune/cluster",
        "taux_cluster_lisse": "risque du cluster",
        "cluster_id": "identifiant cluster"}
lignes = HIST + NEUVES
y = np.arange(len(lignes))
a1 = np.array([100 * IMP1.get(f, 0) for f in lignes])
a3 = np.array([100 * IMP3[f] for f in lignes])

ax[1].barh(y - .21, a1, height=.40, color=GRIS, edgecolor="#fcfcfb",
           linewidth=.7, label="v2 — sans clustering")
ax[1].barh(y + .21, a3, height=.40, color=VERT, edgecolor="#fcfcfb",
           linewidth=.7, label="v3 — avec clustering")
for i, (u, v) in enumerate(zip(a1, a3)):
    if u > .5:
        ax[1].text(u + .7, i - .21, f"{u:.0f}", va="center", fontsize=8, color=MUTED)
    if v > .5:
        ax[1].text(v + .7, i + .21, f"{v:.0f}", va="center", fontsize=8,
                   weight="bold")
ax[1].axhline(len(HIST) - .5, color=INK, lw=1.1, ls=":")
ax[1].set_yticks(y)
ax[1].set_yticklabels([ETIQ[f] for f in lignes], fontsize=8.5)
ax[1].invert_yaxis()
ax[1].set_xlim(0, max(a1.max(), a3.max()) * 1.16)
ax[1].set_xlabel("importance (%)")
ax[1].set_title("Le clustering AJOUTE-t-il, ou REMPLACE-t-il ?",
                fontsize=11.5, weight="bold", loc="left")
ax[1].legend(frameon=False, fontsize=8.5, loc="lower right")

# ── (c) ce que le découpage interne annonçait, ce qu'on a mesuré ─────────
interne = 100 * (CMP.set_index("config").pr_auc_interne["kmeans k=30"]
                 / CMP.set_index("config").pr_auc_interne["aucun"] - 1)
reel = 100 * (V3.pr_auc[0] / V2.pr_auc[0] - 1)
m = ax[2].bar(["Annoncé pendant\nla sélection\n(découpage interne)",
               "Mesuré sur la\nvalidation\n(38 M lignes)"],
              [interne, reel], color=["#c9c4de", VERT],
              edgecolor="#fcfcfb", linewidth=1.2, width=.55)
haut = max(abs(interne), abs(reel))
for r, v in zip(m, [interne, reel]):
    ax[2].text(r.get_x() + r.get_width() / 2,
               v + np.sign(v) * haut * .05 if v else haut * .05,
               f"{v:+.1f} %", ha="center", fontsize=13, weight="bold",
               va="bottom" if v >= 0 else "top")
ax[2].axhline(0, color=INK, lw=1.2)
ax[2].set_ylabel("gain du clustering (%)")
ax[2].set_ylim(min(0, min(interne, reel) * 1.45), haut * 1.55)
ax[2].set_title("Le gain, avant et après vérification", fontsize=11.5,
                weight="bold", loc="left")
ax[2].tick_params(axis="x", labelsize=8.5)

for a in ax:
    a.grid(axis="x" if a is ax[1] else "y", color=GRID, lw=.7)
    a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False); a.tick_params(colors=MUTED)
fig.suptitle("Modèle v3 — validation 2020-2022, 38 M lignes, 9 176 feux (0,0241 %)",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.02)
plt.tight_layout(); plt.show()

print(f"{'meilleure baseline':30s} {BASE_MAX:.4f}")
print(f"{'XGBoost v1':30s} {V1.pr_auc['XGBoost']:.4f}   ×{V1.pr_auc['XGBoost'] / BASE_MAX:.2f}")
print(f"{'XGBoost v2 (Optuna)':30s} {V2.pr_auc[0]:.4f}   ×{V2.pr_auc[0] / BASE_MAX:.2f}")
print(f"{'XGBoost v3 (+ clustering)':30s} {V3.pr_auc[0]:.4f}   ×{V3.pr_auc[0] / BASE_MAX:.2f}")
print(f"\ngain du clustering : {reel:+.1f} %  "
      f"(le découpage interne annonçait {interne:+.2f} %)")
print(f"\nPoids des 4 colonnes de clustering : {100 * IMP3[NEUVES].sum():.1f} %")
rang = {f: list(IMP3.sort_values(ascending=False).index).index(f) + 1 for f in NEUVES}
for f in NEUVES:
    print(f"   {f:24s} {100 * IMP3[f]:5.1f} %   rang {rang[f]}/{len(IMP3)}")

h1, h3 = 100 * IMP1[HIST].sum(), 100 * IMP3[HIST].sum()
n3 = 100 * IMP3[NEUVES].sum()
print(f"\n{'─' * 62}\nAJOUT OU SUBSTITUTION ?\n{'─' * 62}")
print(f"historique brut       {h1:5.1f} %  →  {h3:5.1f} %   ({h3 - h1:+.1f} pts)")
print(f"clustering                  —  →  {n3:5.1f} %   ({n3:+.1f} pts)")
print(f"les deux ensemble     {h1:5.1f} %  →  {h3 + n3:5.1f} %   ({h3 + n3 - h1:+.1f} pts)")
print()
print(f"`taux_commune_lisse` arrive n°1 sur {len(IMP3)} features ({100 * IMP3['taux_commune_lisse']:.1f} %),")
print(f"et `feux_commune_365j` perd {100 * (IMP1['feux_commune_365j'] - IMP3['feux_commune_365j']):.0f} points au même moment.")
print("Le modèle ne découvre donc pas une information neuve : il ADOPTE une")
print("version mieux estimée de celle qu'il avait déjà. D'où un gain réel mais")
print(f"modeste — le net sur « le où » n'est que de {h3 + n3 - h1:+.0f} points.")
print()
print(f"`cluster_id` brut : {100 * IMP3['cluster_id']:.1f} %, rang {rang['cluster_id']}/{len(IMP3)}.")
print("Un entier arbitraire n'a pas d'ordre : un arbre ne peut en tirer que des")
print("découpages sans signification. C'est le TAUX porté par le cluster qui")
print("sert, jamais son numéro.")


## 5. Les trois familles exigées par l'énoncé

Le brief demande **RandomForest, XGBoost et un réseau de neurones**, avec
dropout. Trois familles, la même tâche, les mêmes 52 features, le même
découpage temporel — la comparaison est donc propre.

Trois questions, dans l'ordre :

1. **Laquelle gagne, et de combien ?**
2. **Le dropout sert-il autant sur des arbres que sur un réseau ?** XGBoost a
   un mode `DART` qui éteint des *arbres entiers* pendant la construction :
   c'est la transposition littérale de l'idée. Est-ce que ça marche ?
3. **Les combiner apporte-t-il quelque chose ?**

⚠️ La réponse à la troisième ne dépend **pas** de leur performance
individuelle mais de leur **diversité** : moyenner deux modèles n'aide que
s'ils se trompent sur des lignes *différentes*. Deux modèles excellents qui
font les mêmes erreurs ne donnent, moyennés, que le même modèle en plus lent.
Ça se mesure — c'est la corrélation de rang du panneau central.

Les ensembles sont formés par **moyenne des rangs**, pas des probabilités :
le réseau et les arbres ne sortent pas des scores sur la même échelle, et
moyenner les probabilités brutes laisserait le plus « confiant » dominer pour
une raison sans rapport avec sa justesse.

In [ ]:
"""FIG 4 — Les trois familles, et pourquoi la plus faible est indispensable.

L'énoncé exige RandomForest, XGBoost et un réseau. Trois familles, la même
tâche, les mêmes 52 features, le même découpage. Trois questions :

  1. laquelle gagne, et de combien ?
  2. le dropout — masquer une partie du modèle à l'entraînement — sert-il
     autant sur des arbres que sur un réseau ?
  3. les combiner apporte-t-il quelque chose ?

La réponse à la troisième ne dépend PAS de leur performance individuelle mais
de leur DIVERSITÉ : moyenner deux modèles n'aide que s'ils se trompent sur des
lignes différentes. Ça se mesure — c'est la corrélation de rang.
"""
V1 = pd.read_csv(PROC / "modeles_v1.csv").set_index("modele")
V2 = pd.read_csv(PROC / "modeles_v2.csv")
V3 = pd.read_csv(PROC / "modeles_v3.csv")
MLP = pd.read_csv(PROC / "modeles_mlp.csv")
DART = pd.read_csv(PROC / "modeles_dart.csv")
ENS = pd.read_csv(PROC / "ensembles.csv")
BASE_MAX = BASE.pr_auc.max()
TAUX = 0.0241 / 100

fig, ax = plt.subplots(1, 3, figsize=(16.5, 5.0))

# ── (a) le parcours complet ──────────────────────────────────────────────
etapes = pd.DataFrame({
    "nom": ["Meilleure baseline", "RandomForest", "XGBoost v1", "XGBoost v2\n(Optuna)",
            "XGBoost DART", "MLP\n(dropout)", "XGBoost v3\n(+ clustering)",
            "Ensemble\nXGB v3 + MLP"],
    "v": [BASE_MAX, V1.pr_auc["RandomForest"], V1.pr_auc["XGBoost"], V2.pr_auc[0],
          DART.pr_auc[0], MLP.pr_auc[0], V3.pr_auc[0],
          ENS.set_index("modele").pr_auc["XGBoost v3 + MLP"]],
    "fam": ["baseline", "arbres", "arbres", "arbres", "arbres", "reseau",
            "arbres", "ensemble"],
})
COUL = {"baseline": "#86b6ef", "arbres": BLEU, "reseau": ORANGE, "ensemble": VERT}
b = ax[0].bar(range(len(etapes)), etapes.v, color=[COUL[f] for f in etapes.fam],
              edgecolor="#fcfcfb", linewidth=1.2, width=.72)
for i, v in enumerate(etapes.v):
    ax[0].text(i, v + etapes.v.max() * .022, f"{v:.4f}", ha="center",
               fontsize=8.5, weight="bold")
ax[0].axhline(BASE_MAX, color=INK, ls="--", lw=1.2)
ax[0].set_xticks(range(len(etapes)))
ax[0].set_xticklabels(etapes.nom, fontsize=7.5, rotation=32, ha="right")
ax[0].set_ylabel("PR-AUC sur la validation intégrale")
ax[0].set_ylim(0, etapes.v.max() * 1.16)
ax[0].set_title("Tout le parcours, du hasard à l'ensemble",
                fontsize=11.5, weight="bold", loc="left")

# ── (b) la diversité : la vraie question de l'ensemble ──────────────────
SRC = {"XGBoost v3": ("predictions_val_v3.parquet", "p_xgb_v3"),
       "DART": ("predictions_val_dart.parquet", "p_dart"),
       "MLP": ("predictions_val_mlp.parquet", "p_mlp")}
P = {n: pd.read_parquet(PROC / f, columns=[c])[c].to_numpy(np.float32)
     for n, (f, c) in SRC.items()}
rng = np.random.default_rng(42)
s = rng.choice(len(P["MLP"]), 500_000, replace=False)


def rangs(v):
    o = np.empty(len(v), np.int64)
    o[np.argsort(v, kind="stable")] = np.arange(len(v))
    return o / len(v)


R = {n: rangs(v[s]) for n, v in P.items()}
noms = list(SRC)
C = np.array([[np.corrcoef(R[a], R[b])[0, 1] for b in noms] for a in noms])
im = ax[1].imshow(C, cmap="RdYlGn_r", vmin=.96, vmax=1.0)
for i in range(3):
    for j in range(3):
        ax[1].text(j, i, f"{C[i, j]:.4f}", ha="center", va="center",
                   fontsize=11, weight="bold",
                   color="#fcfcfb" if C[i, j] > .985 else INK)
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(noms, fontsize=9)
ax[1].set_yticks(range(3)); ax[1].set_yticklabels(noms, fontsize=9)
ax[1].tick_params(colors=MUTED, length=0)
ax[1].spines[:].set_visible(False)
ax[1].set_title("Corrélation de rang — plus c'est BAS,\nplus il y a à gagner",
                fontsize=11.5, weight="bold", loc="left")

# ── (c) ce que chaque combinaison rapporte ──────────────────────────────
ref = V3.pr_auc[0]
E = ENS.sort_values("pr_auc")
c = [VERT if g > 0 else ROUGE for g in E.gain_vs_v3_pct]
ax[2].barh(range(len(E)), E.gain_vs_v3_pct, color=c, edgecolor="#fcfcfb",
           linewidth=1, height=.68)
for i, (g, a) in enumerate(zip(E.gain_vs_v3_pct, E.pr_auc)):
    dx = .08 if g > 0 else -.08
    ax[2].text(g + dx, i, f"{g:+.2f} %", va="center",
               ha="left" if g > 0 else "right", fontsize=9.5, weight="bold")
ax[2].axvline(0, color=INK, lw=1.4)
ax[2].set_yticks(range(len(E)))
ax[2].set_yticklabels([n.replace(" + ", "\n+ ") for n in E.modele], fontsize=8)
ax[2].set_xlabel("gain sur le meilleur modèle seul (XGBoost v3), en %")
ax[2].set_xlim(E.gain_vs_v3_pct.min() - .55, E.gain_vs_v3_pct.max() + .75)
ax[2].set_title("Combiner : ce que ça rapporte",
                fontsize=11.5, weight="bold", loc="left")

for a in (ax[0], ax[2]):
    a.grid(axis="y" if a is ax[0] else "x", color=GRID, lw=.7)
    a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False); a.tick_params(colors=MUTED)
fig.suptitle("Les trois familles exigées par l'énoncé — validation 2020-2022, "
             "38 M lignes, 9 176 feux",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.02)
plt.tight_layout(); plt.show()

print(f"{'modèle':34s} {'PR-AUC':>9s} {'lift':>8s}")
print("─" * 54)
for n, v in zip(etapes.nom, etapes.v):
    print(f"{n.replace(chr(10), ' '):34s} {v:9.4f} {v / TAUX:7.1f}×")

print(f"\n{'─' * 66}\nCE QUE ÇA DIT\n{'─' * 66}")
print("1. LES TROIS FAMILLES TIENNENT DANS 2 %.")
print(f"   XGBoost {V3.pr_auc[0]:.4f}, DART {DART.pr_auc[0]:.4f}, MLP {MLP.pr_auc[0]:.4f}.")
print("   Le passage baseline → features avait rapporté +63 %. La famille")
print("   d'algorithme en vaut 2. Le plafond est fixé par les DONNÉES.")
print()
print("2. LE DROPOUT NE SE TRANSPOSE PAS AUX ARBRES.")
print(f"   DART éteint des arbres entiers : {100 * (DART.pr_auc[0] / V3.pr_auc[0] - 1):+.1f} % — et ~5× plus lent.")
print("   Optuna avait déjà réglé la sur-spécialisation en divisant le")
print("   learning rate par quatre. DART arrive après la bataille.")
print("   Sur le réseau en revanche, le dropout est le 2e hyperparamètre")
print("   sur six (31,8 % de la variation), retenu à 0,52.")
print()
print("3. LE MODÈLE LE PLUS FAIBLE EST LE SEUL QUI SERVE À L'ENSEMBLE.")
i_d = noms.index("DART"); i_m = noms.index("MLP"); i_x = noms.index("XGBoost v3")
print(f"   XGBoost ↔ DART {C[i_x, i_d]:.4f} — deux ensembles d'arbres, même biais.")
print(f"   XGBoost ↔ MLP  {C[i_x, i_m]:.4f} — arbre contre réseau, vraie diversité.")
g = ENS.set_index("modele").gain_vs_v3_pct
print(f"   D'où : XGB + DART {g['XGBoost v3 + DART']:+.2f} %, "
      f"XGB + MLP {g['XGBoost v3 + MLP']:+.2f} %.")
print("   C'est la DIVERSITÉ qui paie, pas la performance individuelle.")


## 5. Synthèse

### Les scores

| Prédicteur | PR-AUC | lift | vs baseline |
|---|---|---|---|
| Hasard | 0,0002 | 1,0× | — |
| Meilleure baseline (Historique × EFFIS) | 0,0101 | 42,1× | ×1,00 |
| XGBoost v1 (réglé à la main) | 0,0166 | 68,8× | ×1,63 |
| XGBoost v2 (Optuna) | 0,0175 | 72,4× | ×1,72 |
| **XGBoost v3 (+ clustering)** | **0,0177** | **73,4×** | **×1,74** |

Le clustering apporte **+1,3 %** sur le v2. Réel, mais modeste — et le
pourquoi est plus instructif que le combien.

### Le résultat central : substitution, pas addition

`taux_commune_lisse` arrive **n°1 sur 52 features**, à 25,3 % de l'importance.
Les quatre colonnes de clustering pèsent ensemble **33,7 %**. Et pourtant la
PR-AUC ne bouge que de 1,3 %. La contradiction n'est qu'apparente :

| | v2 | v3 | variation |
|---|---|---|---|
| historique brut (5 features) | 54,6 % | 29,3 % | **−25,2 pts** |
| clustering (4 features) | — | 33,7 % | +33,7 pts |
| **« le où », au total** | **54,6 %** | **63,0 %** | **+8,4 pts** |

`feux_commune_365j` s'effondre de 35,2 % à 11,1 % au moment précis où
`taux_commune_lisse` prend 25,3 %. **Le modèle ne découvre pas une information
neuve : il adopte une version mieux estimée de celle qu'il avait déjà.** Le net
sur l'information spatiale n'est que de +8,4 points — d'où le gain modeste.

C'est cohérent avec le diagnostic du v1 : le trou n'était pas l'absence de
signal spatial, c'était la *qualité d'estimation* de ce signal pour les
communes rares.

### Ce que valait le trou, en chiffres

**27 938 communes — 80,4 % du territoire — n'ont jamais brûlé sur 2006-2019.**
Pour le v1 elles étaient rigoureusement identiques : quatre features
d'historique, quatre zéros, de la garrigue varoise au bocage normand.

Le lissage les sépare d'un facteur **×1512**. Mais attention à la lecture :
n'ayant aucun feu et toutes exactement le même nombre de jours, chacune hérite
du taux de son cluster — il n'existe donc que **31 valeurs distinctes** pour ces
27 938 communes. La résolution vient entièrement du nombre de clusters.

### La typologie tient debout toute seule

| | Cluster | Ce qui le caractérise | Risque quotidien |
|---|---|---|---|
| le plus exposé | c28, 160 communes | maquis **+4,2 σ**, littoral, peu agricole | 0,3266 % |
| le moins exposé | c6, 4 957 communes | agricole +1,8 σ, plat, peu boisé | 0,0002 % |

×1512 d'écart entre des groupes **formés sans jamais regarder où le feu tombe**.
Ce n'est pas une tautologie : c'est la mesure de ce que la composition du
territoire explique à elle seule, avant toute météo.

### `cluster_id` brut ne sert à rien — et c'était prévisible

0,4 % d'importance, **rang 43 sur 52**. Un entier arbitraire n'a pas d'ordre :
un arbre ne peut en tirer que des découpages du type « cluster_id < 12,5 », qui
ne signifient rien. C'est le **taux** porté par le cluster qui travaille, jamais
son numéro. La colonne est conservée pour la traçabilité et pour SHAP, pas pour
sa contribution.

### Encore une fois, le découpage interne a sous-estimé

| Où le gain a été mesuré | Gain annoncé |
|---|---|
| pendant la sélection — découpage interne | +0,20 % |
| après coup — validation intégrale | **+1,3 %** |

Facteur ~6, même sens et même ordre de grandeur que pour Optuna (+0,7 % annoncé,
+5,2 % mesuré). Ce n'est plus une surprise, c'est un **biais systématique** de
ce proxy : à 8 % de positifs la PR-AUC dépend de tout le classement, à 0,024 %
elle ne dépend que de son extrême sommet.

→ **À retenir pour la suite du projet** : tout score mesuré sur le découpage
interne doit être lu comme un classement entre configurations, jamais comme une
prévision du gain. Le facteur de sous-estimation observé est de 6 à 8.

### Prochaines étapes

1. **MLP**, la troisième famille exigée par l'énoncé — `sklearn.MLPClassifier`
   n'a pas de dropout, arbitrage torch/keras à trancher
2. **SHAP** sur le v3, avec la matrice de corrélation sous les yeux : entre
   `taux_commune_lisse` et `feux_commune_365j`, les importances des features
   corrélées seront mal lues sans elle
3. **Calibration** du v3 sur la validation, puis **évaluation test — une seule
   fois**
4. **Application Streamlit**, Docker, CI/CD, DVC